# CS5014 Machine Learning 

### Practical 1


##### Credits: 50% of the coursework

## Aims

The objectives of this assignment are:

* deepen your understanding of linear regression and logistic regression
* gain experience in implementing learning algorithms 
* gain experience in evaluating machine learning algorithms
* gain experience in hyper-parameter tuning


## Set-up

You are **only allowed** to use the following imported packages for this practical. No off-the-shelf machine learning packages such as _scikit-learn_ are allowed. 


In [ ]:
# if you use jupyter-lab, switch to %matplotlib inline instead
%matplotlib inline
# %matplotlib notebook
%config Completer.use_jedi = False
import matplotlib.pyplot as plt
import autograd.numpy as np  # Thinly-wrapped numpy
from autograd import grad    # The only autograd function you may ever need
import autograd.numpy.linalg as linalg
import matplotlib.pyplot as plt
import pandas as pd

## Question 1 (Logistic regression)

In this question, we are going to implement an logistic regression model to do binary classification on a simulated dataset. The dataset's input feature are four-dimensional vectors $\mathbf{x}^{(i)} \in \mathbb{R}^4$ and as expected the target $y^{(i)} \in \{0, 1\}$. 


The dataset $\{\mathbf{x}^{(i)}, y^{(i)}\}$ is imported below for you:
* ``dataset1``: 2000 observations and each input $\mathbf{x}$ has 4 features 
* and the last column is the target ${y}^{(i)}$
* the dataset is then split into training and testing parts

In [ ]:
# read in dataset1
dataset1_df = pd.read_csv('./datasets/dataset1.csv', header=0)
dataset1 = np.array(dataset1_df)
d1X, d1Y = dataset1[:, 0:4], dataset1[:, -1]
# split the data into training and testing 
# the training dataset has the first 1500 observation; 
# in practice, you should randomly shuffle before the split
d1_xtrain, d1_ytrain = d1X[0:1500, :], d1Y[0:1500]
# the testing dataset has the last 500
d1_xtest, d1_ytest = d1X[1500:, :], d1Y[1500:]

As suggested in the lecture, it is convenient to introduce dummy ones to avoid learning the bias separately.

In [ ]:
d1_xtrain_dummy = np.column_stack((np.ones(d1_xtrain.shape[0]), d1_xtrain))

### Task 1.1 Implementation of logistic regression

Your task here is to implement a gradient descent based algorithm to train a logistic regression model. For this task, you cannot use `autograd`'s auto-differentiation method (*i.e.* the imported `grad` method). You will be guided to finish the task step by step. 

First, implement the `sigmoid` function:

$$\sigma(z) = \frac{1}{1+e^{-z}}$$

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

Second, implement the cross-entropy loss and its gradient. You may want to refer to the lecture slides for the details. Recall the binary **C**ross **E**ntropy (CE) _loss_ is 


$$
L(\mathbf{w})=  \frac{1}{n}\sum_{i=1}^n -{y^{(i)}} \ln \sigma^{(i)}- (1- y^{(i)}) \ln (1-\sigma^{(i)})
$$

where $\sigma^{(i)} =\sigma(\mathbf{w}^\top\mathbf{x}^{(i)} + b).$

In [ ]:
def cross_entropy_loss(w, X, y):
    n = X.shape[0]
    sigma = sigmoid(X @ w)
    sigma = np.clip(sigma, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(sigma) + (1 - y) * np.log(1 - sigma))


In [ ]:
def gradient_ce_loss(w, X, y):
    n = X.shape[0]
    sigma = sigmoid(X @ w)
    return (1.0 / n) * (X.T @ (sigma - y))


Lastly, implement the gradient descent algorithm.

In [ ]:
def logistic_regression_train(X, y, gamma, tol=1e-4, maxIters=100):
    n, d = X.shape
    # initialise w0
    w0 = np.zeros(d)
    losses = []
    # loop until converge
    for i in range(maxIters):
        loss = cross_entropy_loss(w0, X, y)
        losses.append(loss)
        g = gradient_ce_loss(w0, X, y)
        w0 = w0 - gamma * g
        # check convergence: if loss change is below tolerance, stop
        if len(losses) > 1 and abs(losses[-2] - losses[-1]) < tol:
            break
    return w0, losses


After you finish implementing all the above methods, use your learning algorithm train a logistic regression model on the training dataset and answer the following questions:

* plot the learning curve
* how did you check whether the learning has converged ?
* report the learning rate parameter used 
* report the learnt parameter $\mathbf{w}$ and bias $b$

In [ ]:
# Train the model
gamma = 0.1  # learning rate
w_learned, losses = logistic_regression_train(d1_xtrain_dummy, d1_ytrain, gamma, tol=1e-6, maxIters=1000)

# Plot the learning curve
plt.figure()
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("Cross-Entropy Loss")
plt.title("Learning Curve")
plt.show()

# Convergence check: training stops when the absolute change in loss between
# consecutive iterations is less than the tolerance (1e-6).

print(f"Learning rate (gamma): {gamma}")
print(f"Number of iterations: {len(losses)}")
print(f"Final loss: {losses[-1]:.6f}")
print(f"Learnt bias b = w[0]: {w_learned[0]:.6f}")
print(f"Learnt weights w[1:]: {w_learned[1:]}")


### Task 1.2 Testing performance

Implement a prediction method that takes as input the features together with the learnt parameter and output the predicted labels.

In [ ]:
def predict_logistic_regression(w, X=d1_xtest):
    n, d = X.shape
    # Add dummy ones column for bias
    X_dummy = np.column_stack((np.ones(n), X))
    probs = sigmoid(X_dummy @ w)
    return (probs >= 0.5).astype(float)


Report the test performance on the unseen test dataset.

In [ ]:
y_pred = predict_logistic_regression(w_learned, d1_xtest)
accuracy = np.mean(y_pred == d1_ytest)
print(f"Test accuracy: {accuracy:.4f}")

### Task 1.3 Regularisation

In this sub-task, you are going to apply $L_2$ regularisation to the logistic regression model. The regularised loss is

$$
L(\mathbf{w})=  \frac{1}{n}\sum_{i=1}^n -{y^{(i)}} \ln \sigma^{(i)}- (1- y^{(i)}) \ln (1-\sigma^{(i)}) + \frac{\lambda}{2} \mathbf{w}^\top\mathbf{w}
$$

* where $\lambda >0$ is the regularisation hyperparameter

* note that we do not usually apply penalty on the bias parameter $b$

Implement the following method that fits a regularised logistic regression model with a given $\lambda$.

In [ ]:
def logistic_regression_reg_train(X, y, gamma, lam=1.0, tol=1e-4, maxIters=100):
    n, d = X.shape
    # initialise w0
    w0 = np.zeros(d)
    losses = []
    # loop until converge
    for i in range(maxIters):
        sigma = sigmoid(X @ w0)
        sigma = np.clip(sigma, 1e-12, 1 - 1e-12)
        # Regularised CE loss (do NOT penalise bias w0[0])
        ce = -np.mean(y * np.log(sigma) + (1 - y) * np.log(1 - sigma))
        reg = (lam / 2.0) * np.dot(w0[1:], w0[1:])
        loss = ce + reg
        losses.append(loss)
        # Gradient
        g = (1.0 / n) * (X.T @ (sigma - y))
        # Add regularisation gradient (not for bias)
        reg_grad = lam * np.concatenate(([0.0], w0[1:]))
        g = g + reg_grad
        w0 = w0 - gamma * g
        if len(losses) > 1 and abs(losses[-2] - losses[-1]) < tol:
            break
    return w0, losses


Complete and report the following two results
* report the training loss by setting $\lambda=0.03$
* report the testing performance for the regularised logistic regression model with $\lambda=0.03$

In [ ]:
w_reg, losses_reg = logistic_regression_reg_train(d1_xtrain_dummy, d1_ytrain, gamma=0.1, lam=0.03, tol=1e-6, maxIters=1000)
print(f"Training loss (lambda=0.03): {losses_reg[-1]:.6f}")

y_pred_reg = predict_logistic_regression(w_reg, d1_xtest)
accuracy_reg = np.mean(y_pred_reg == d1_ytest)
print(f"Test accuracy (lambda=0.03): {accuracy_reg:.4f}")


### Task 1.4 Cross-validation

Use K-fold cross-validation (K=5) to choose the optimal $\lambda$. You should use the cross entropy loss as the selection criteria. The candidate hyper-parameter set for $\lambda$ should be 10 numbers between $10^{-3} = 0.001$ and $10^{0}=1$ (*i.e.* 10 numbers in log space). The candidate set is listed below. 

What is the optimal $\lambda$?

In [ ]:
lambda_set = np.logspace(-3, 0, 10)
K = 5
n_train = d1_xtrain_dummy.shape[0]
fold_size = n_train // K

avg_losses = []

for lam in lambda_set:
    fold_losses = []
    for k in range(K):
        # Split into validation and training folds
        val_start = k * fold_size
        val_end = (k + 1) * fold_size
        
        X_val = d1_xtrain_dummy[val_start:val_end]
        y_val = d1_ytrain[val_start:val_end]
        
        X_tr = np.concatenate([d1_xtrain_dummy[:val_start], d1_xtrain_dummy[val_end:]], axis=0)
        y_tr = np.concatenate([d1_ytrain[:val_start], d1_ytrain[val_end:]], axis=0)
        
        w_cv, _ = logistic_regression_reg_train(X_tr, y_tr, gamma=0.1, lam=lam, tol=1e-6, maxIters=1000)
        
        # Evaluate on validation fold using CE loss (no regularisation term)
        val_loss = cross_entropy_loss(w_cv, X_val, y_val)
        fold_losses.append(val_loss)
    
    avg_losses.append(np.mean(fold_losses))

# Find optimal lambda
best_idx = np.argmin(avg_losses)
best_lambda = lambda_set[best_idx]
print(f"Optimal lambda: {best_lambda:.6f}")

# Plot CV results
plt.figure()
plt.plot(lambda_set, avg_losses, 'o-')
plt.xscale('log')
plt.xlabel("Lambda")
plt.ylabel("Average CV Loss")
plt.title("5-Fold CV: Lambda Selection")
plt.show()


### Task 1.5 Stochastic gradient descent 


Implement a stochastic gradient descent algorithm with mini-batch size of 1. You should consider shuffling the training dataset to improve the convergence speed.

In [ ]:
def logistic_regression_reg_sgd_train(X, y, gamma, lam=1.0, tol=1e-4, maxIters=100):
    n, d = X.shape
    # initialise w0
    w0 = np.zeros(d)
    losses = []
    # loop until converge
    for epoch in range(maxIters):
        # Shuffle the data at the start of each epoch
        perm = np.random.permutation(n)
        X_shuffled = X[perm]
        y_shuffled = y[perm]
        
        # for each observation (x^i, y^i) in (X, Y)
        for i in range(n):
            xi = X_shuffled[i:i+1]  # keep 2D shape (1, d)
            yi = y_shuffled[i:i+1]
            # compute gradient and apply gradient descent
            sigma_i = sigmoid(xi @ w0)
            g = xi.T @ (sigma_i - yi)  # gradient for single sample (no 1/n since n=1)
            reg_grad = lam * np.concatenate(([0.0], w0[1:]))
            g = g.flatten() + reg_grad
            w0 = w0 - gamma * g
        
        # Record loss after each epoch
        loss = cross_entropy_loss(w0, X, y) + (lam / 2.0) * np.dot(w0[1:], w0[1:])
        losses.append(loss)
        
        if len(losses) > 1 and abs(losses[-2] - losses[-1]) < tol:
            break

    # Plot the loss
    plt.plot(losses)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss over epochs")
    plt.show()

    # Print the final loss
    print("Final loss:", losses[-1])
    
    return w0, losses


In [ ]:
# Train with SGD using the optimal lambda from CV and compare with batch GD
w_sgd, losses_sgd = logistic_regression_reg_sgd_train(
    d1_xtrain_dummy, d1_ytrain, gamma=0.01, lam=best_lambda, tol=1e-6, maxIters=50
)

# Test performance
y_pred_sgd = predict_logistic_regression(w_sgd, d1_xtest)
accuracy_sgd = np.mean(y_pred_sgd == d1_ytest)
print(f"SGD Test accuracy (lambda={best_lambda:.6f}): {accuracy_sgd:.4f}")
print(f"Batch GD Test accuracy (lambda=0.03): {accuracy_reg:.4f}")
print(f"SGD weights: {w_sgd}")


## Question 2 (Fixed basis expansion)



In this question, we are going to implement a fixed basis expansion regression model. The dataset for this question has a univariate feature $\mathbf{x}^{(i)} \in \mathbb{R}$ and as expected the target $y^{(i)} \in \mathbb{R}$ is real valued. 


The dataset is imported below for you:
* ``dataset2``: 1000 observations and each input ${x}^{(i)}$ is a scalar 
* and the last column is the target ${y}^{(i)}$
* the dataset is then split into training and testing parts

In [ ]:
# read in dataset2
dataset2_df = pd.read_csv('./datasets/dataset2.csv', header=0)
dataset2 = np.array(dataset2_df)
d2X, d2Y = dataset2[:, 0], dataset2[:, -1]
# split the data into training and testing 
# the training dataset has the first 800 observation; 
# in practice, you should randomly shuffle before the split
d2_xtrain, d2_ytrain = d2X[0:800], d2Y[0:800]
# # the testing dataset has the last 500
d2_xtest, d2_ytest = d2X[800:], d2Y[800:]

The data is plotted below.

In [ ]:
plt.scatter(d2_xtrain, d2_ytrain,  c ="blue", s=1.5)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

### Task 2.1 Basis function

Implement the radian-basis-function (rbf), 

$$\phi(x; \mu, s) = \exp\left \{- \frac{(x-\mu)^2}{2s} \right \}$$

In [ ]:
def phi_rbf(x, mu, s):
    return np.exp(-((x - mu) ** 2) / (2 * s))

### Task 2.2 Fixed basis expansion regression

Implement the fixed basis expansion regression model. Specifically, you should 
* first apply $K$ fixed basis expansion on the input $\{x^{(i)}\}$ to form the expanded design matrix $\mathbf{\Phi}$
* then fit a regression model by a non-iterative algorithm (*i.e.* the normal equation method)

* you are free to choose the $K$ expansion locations, some possible choices
  * select  locations with evenly spaced intervals between $x$'s range 
  * randomly choose  observations from the  training data 
  * randomly select $K$ points within $\{x^{(i)}\}$'s range

In [ ]:
def fixed_basis_rbf_reg(X, y, mus, s):
    n = X.shape[0]
    K = len(mus)

    # Build the design matrix: column of ones + K RBF columns
    Phi = np.ones((n, K + 1))
    for j in range(K):
        Phi[:, j + 1] = phi_rbf(X, mus[j], s)

    # Solve via the normal equation
    w = linalg.solve(Phi.T @ Phi, Phi.T @ y)
    return w

Implement a `predict` method, that output the prediction $\hat{y}$ given input $x_{test}$
* ideally, your method should be able to predict multiple input at the same time (vectorised)

* plot the fitted function on top of the scatter plot of the training data

In [ ]:
# Choose K evenly spaced centres across the range of training data
K = 10
mus = np.linspace(d2_xtrain.min(), d2_xtrain.max(), K)

# s controls the width of each RBF — a reasonable starting point
# is the squared spacing between centres
s = ((mus[1] - mus[0]) ** 2) / 2.0

# Fit the model
w_rbf = fixed_basis_rbf_reg(d2_xtrain, d2_ytrain, mus, s)

def predict_rbf_reg(X, w, mus, s):
    n = X.shape[0]
    K = len(mus)
    Phi = np.ones((n, K + 1))
    for j in range(K):
        Phi[:, j + 1] = phi_rbf(X, mus[j], s)
    return Phi @ w

In [ ]:
# Generate a fine grid of x values for a smooth curve
x_plot = np.linspace(d2_xtrain.min(), d2_xtrain.max(), 300)
y_plot = predict_rbf_reg(x_plot, w_rbf, mus, s)

plt.figure()
plt.scatter(d2_xtrain, d2_ytrain, c="blue", s=1.5, label="Training data")
plt.plot(x_plot, y_plot, c="red", linewidth=2, label="RBF fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Fixed Basis RBF Regression (K={K})")
plt.legend()
plt.show()

In [ ]:
y_test_pred = predict_rbf_reg(d2_xtest, w_rbf, mus, s)
mse = np.mean((y_test_pred - d2_ytest) ** 2)
print(f"Test MSE: {mse:.4f}")

### Task 2.3 Other basis function


Implement another basis function of your choice and fit the regression model. Plot the fitted result below.

In [ ]:
def phi_fourier(x, k, func='sin'):
    if func == 'sin':
        return np.sin(2 * np.pi * k * x)
    else:
        return np.cos(2 * np.pi * k * x)

def build_fourier_design_matrix(X, K):
    n = X.shape[0]
    Phi = np.ones((n, 2 * K + 1))
    for k in range(1, K + 1):
        Phi[:, 2 * k - 1] = phi_fourier(X, k, 'sin')
        Phi[:, 2 * k]     = phi_fourier(X, k, 'cos')
    return Phi

def fixed_basis_fourier_reg(X, y, K):
    Phi = build_fourier_design_matrix(X, K)
    w = linalg.solve(Phi.T @ Phi, Phi.T @ y)
    return w

def predict_fourier_reg(X, w, K):
    Phi = build_fourier_design_matrix(X, K)
    return Phi @ w

In [ ]:
# =====================================================
# Plot 1: Fitted curve on top of training data
# =====================================================
K = 10  # or whichever K you settled on
w_fourier = fixed_basis_fourier_reg(d2_xtrain, d2_ytrain, K)

x_plot = np.linspace(d2_xtrain.min() - 0.02, d2_xtrain.max() + 0.02, 500)
y_plot = predict_fourier_reg(x_plot, w_fourier, K)

# Also get RBF predictions for comparison
K_rbf = 10
mus = np.linspace(d2_xtrain.min(), d2_xtrain.max(), K_rbf)
s = ((mus[1] - mus[0]) ** 2) / 2.0
w_rbf = fixed_basis_rbf_reg(d2_xtrain, d2_ytrain, mus, s)
y_rbf_plot = predict_rbf_reg(x_plot, w_rbf, mus, s)

plt.figure(figsize=(10, 6))
plt.scatter(d2_xtrain, d2_ytrain, c='blue', s=2, alpha=0.3, label='Training data')
plt.plot(x_plot, y_plot, c='red', linewidth=2, label=f'Fourier (K={K})')
plt.plot(x_plot, y_rbf_plot, c='green', linewidth=2, linestyle='--', label=f'RBF (K={K_rbf})')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Task 2.3: Fourier vs RBF Basis Regression")
plt.legend()
plt.show()

# =====================================================
# Plot 2: Multiple K values to show underfitting → good fit
# =====================================================
K_values = [2, 4, 6, 8, 10, 15]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, K_val in enumerate(K_values):
    ax = axes[idx]
    w_f = fixed_basis_fourier_reg(d2_xtrain, d2_ytrain, K_val)
    y_f_plot = predict_fourier_reg(x_plot, w_f, K_val)
    
    y_test_pred = predict_fourier_reg(d2_xtest, w_f, K_val)
    test_mse = np.mean((y_test_pred - d2_ytest) ** 2)
    
    ax.scatter(d2_xtrain, d2_ytrain, c='blue', s=1.5, alpha=0.3, label='Train data')
    ax.plot(x_plot, y_f_plot, c='red', linewidth=2, label=f'Fourier K={K_val}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Fourier K={K_val} ({2*K_val+1} params)\nTest MSE={test_mse:.4f}')
    ax.legend(fontsize=8)
    ax.set_ylim(-5, 10)

plt.suptitle('Task 2.3: Fourier Basis Regression — Effect of K', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# =====================================================
# Plot 3: Illustration of the Fourier basis functions
# =====================================================
x_basis = np.linspace(0, 1, 300)

plt.figure(figsize=(10, 5))
plt.axhline(y=1, color='black', linewidth=1.5, linestyle='-', label='φ₀ = 1')
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for k in range(1, 5):
    plt.plot(x_basis, np.sin(2 * np.pi * k * x_basis), color=colors[2*k-1],
             linewidth=1.5, label=f'sin(2π·{k}·x)')
    plt.plot(x_basis, np.cos(2 * np.pi * k * x_basis), color=colors[2*k],
             linewidth=1.5, linestyle='--', label=f'cos(2π·{k}·x)')
plt.xlabel('x')
plt.ylabel('φ(x)')
plt.title('Fourier Basis Functions (first 4 frequency pairs)')
plt.legend(fontsize=8, ncol=3, loc='upper right')
plt.tight_layout()
plt.show()

### Task 2.4 Advanced task (extension*)

We assume the noise scale is a constant for ordinary linear regresssion model. However, the noise scale for this dataset increases as $x$ gets larger. This is known as heteroscedasticity. Fit a fixed basis regression model that can also learn the heteroscedasticity. Your are allowed to use auto-differentiation for this question.

In [ ]:
def build_rbf_design_matrix(X, mus, s):
    """Build design matrix with intercept + K RBF columns. Shape: (n, K+1)"""
    n = X.shape[0]
    K = len(mus)
    Phi = np.ones((n, K + 1))
    for j in range(K):
        Phi[:, j + 1] = phi_rbf(X, mus[j], s)
    return Phi

def nll_heteroscedastic(theta, Phi, y, K_plus_1):
    w_mu = theta[:K_plus_1]
    w_sigma = theta[K_plus_1:]
    mu = Phi @ w_mu
    log_var = Phi @ w_sigma
    var = np.exp(log_var)
    residuals = y - mu
    nll = np.mean(0.5 * log_var + (residuals ** 2) / (2.0 * var))
    return nll

def train_heteroscedastic(X, y, mus, s, gamma=0.05, maxIters=5000, tol=1e-8):
    Phi = build_rbf_design_matrix(X, mus, s)
    K_plus_1 = Phi.shape[1]
    theta = np.zeros(2 * K_plus_1)
    grad_nll = grad(nll_heteroscedastic, argnum=0)
    losses = []
    for i in range(maxIters):
        loss = nll_heteroscedastic(theta, Phi, y, K_plus_1)
        losses.append(float(loss))
        g = grad_nll(theta, Phi, y, K_plus_1)
        theta = theta - gamma * g
        if len(losses) > 1 and abs(losses[-2] - losses[-1]) < tol:
            break
    w_mu = theta[:K_plus_1]
    w_sigma = theta[K_plus_1:]
    return w_mu, w_sigma, losses

def predict_heteroscedastic(X, w_mu, w_sigma, mus, s):
    Phi = build_rbf_design_matrix(X, mus, s)
    mu = Phi @ w_mu
    log_var = Phi @ w_sigma
    sigma = np.sqrt(np.exp(log_var))
    return mu, sigma

# Build basis and train
K = 10
mus = np.linspace(d2_xtrain.min(), d2_xtrain.max(), K)
s = ((mus[1] - mus[0]) ** 2) / 2.0

w_mu, w_sigma, losses = train_heteroscedastic(
    d2_xtrain, d2_ytrain, mus, s, gamma=0.05, maxIters=5000
)



In [ ]:
# =====================================================
# Train the models first (assumes build_rbf_design_matrix, 
# train_heteroscedastic, predict_heteroscedastic are defined)
# =====================================================
K = 10
mus = np.linspace(d2_xtrain.min(), d2_xtrain.max(), K)
s = ((mus[1] - mus[0]) ** 2) / 2.0

w_mu, w_sigma, losses = train_heteroscedastic(
    d2_xtrain, d2_ytrain, mus, s, gamma=0.05, maxIters=5000
)

# Homoscedastic baseline for comparison
w_homo = fixed_basis_rbf_reg(d2_xtrain, d2_ytrain, mus, s)
Phi_train = build_rbf_design_matrix(d2_xtrain, mus, s)
residuals_homo = d2_ytrain - Phi_train @ w_homo
sigma_homo = np.std(residuals_homo)

# Predictions on a fine grid
x_plot = np.linspace(d2_xtrain.min() - 0.01, d2_xtrain.max() + 0.01, 500)
mu_plot, sigma_plot = predict_heteroscedastic(x_plot, w_mu, w_sigma, mus, s)
Phi_plot = build_rbf_design_matrix(x_plot, mus, s)
mu_homo_plot = Phi_plot @ w_homo

# =====================================================
# Plot 1: Learning curve
# =====================================================
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Negative Log-Likelihood')
plt.title('Heteroscedastic Model: Learning Curve')
plt.show()

# =====================================================
# Plot 2: Side-by-side — Heteroscedastic vs Homoscedastic
# =====================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Heteroscedastic
ax = axes[0]
ax.scatter(d2_xtrain, d2_ytrain, c='blue', s=2, alpha=0.3, label='Training data')
ax.plot(x_plot, mu_plot, c='red', linewidth=2, label='μ(x) — predicted mean')
ax.fill_between(x_plot, mu_plot - 1.96 * sigma_plot, mu_plot + 1.96 * sigma_plot,
                color='red', alpha=0.15, label='±1.96σ(x) — 95% CI')
ax.fill_between(x_plot, mu_plot - sigma_plot, mu_plot + sigma_plot,
                color='red', alpha=0.25, label='±1σ(x) — 68% CI')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Heteroscedastic Model\n(learned varying noise)')
ax.legend(fontsize=9)
ax.set_ylim(-6, 12)

# Right: Homoscedastic
ax = axes[1]
ax.scatter(d2_xtrain, d2_ytrain, c='blue', s=2, alpha=0.3, label='Training data')
ax.plot(x_plot, mu_homo_plot, c='green', linewidth=2, label='μ(x) — predicted mean')
ax.fill_between(x_plot, mu_homo_plot - 1.96 * sigma_homo, mu_homo_plot + 1.96 * sigma_homo,
                color='green', alpha=0.15, label=f'±1.96σ — 95% CI (σ={sigma_homo:.2f})')
ax.fill_between(x_plot, mu_homo_plot - sigma_homo, mu_homo_plot + sigma_homo,
                color='green', alpha=0.25, label='±1σ — 68% CI')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Standard (Homoscedastic) Model\n(constant noise assumed)')
ax.legend(fontsize=9)
ax.set_ylim(-6, 12)

plt.suptitle('Task 2.4: Heteroscedastic vs Homoscedastic Regression', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# =====================================================
# Plot 3: Learned noise function σ(x) vs empirical
# =====================================================
plt.figure(figsize=(8, 5))
plt.plot(x_plot, sigma_plot, c='red', linewidth=2, label='σ(x) — learned noise')
plt.axhline(y=sigma_homo, color='green', linewidth=2, linestyle='--',
            label=f'σ = {sigma_homo:.2f} — constant (homoscedastic)')

# Overlay empirical bin standard deviations
n_bins = 15
sorted_idx = np.argsort(d2_xtrain)
xs = d2_xtrain[sorted_idx]
ys = d2_ytrain[sorted_idx]
bin_edges = np.linspace(xs.min(), xs.max(), n_bins + 1)
for i in range(n_bins):
    mask = (xs >= bin_edges[i]) & (xs < bin_edges[i+1])
    if mask.sum() > 2:
        center = (bin_edges[i] + bin_edges[i+1]) / 2
        empirical_std = ys[mask].std()
        plt.scatter(center, empirical_std, c='blue', s=40, zorder=5, marker='D',
                    label='Empirical bin std' if i == 0 else None)

plt.xlabel('x')
plt.ylabel('σ(x)')
plt.title('Learned Noise Function vs Empirical Standard Deviation')
plt.legend()
plt.show()

## Submission
Hand in via MMS: the completed jupyter notebook. Your notebook should be reproducible.



## Marking
Your submission will be marked as a whole. 

* to get a grade above 7, you are expected to finish at least Task 1.1-1.2 to a good standard
* to get a grade above 10 and up to 13, you are expected to complete Task 1.1-1.4 to a good standard
* to get a grade above 13 and up to 17, you are expected to complete all tasks except 2.3 and 2.4 to a good standard
* to achieve a grade of 17-18, you are expected to finish all tasks except Task 2.4 flawlessly 
* to get 18+, you are expected to attempt all questions flawlessly


Marking is according to the standard mark descriptors published in the Student Handbook at:

https://info.cs.st-andrews.ac.uk/student-handbook/learning-teaching/feedback.html#GeneralMarkDescriptors


You must reference any external sources used. Guidelines for good academic practice are outlined in the student handbook at https://info.cs.st-andrews.ac.uk/student-handbook/academic/gap.html
